# Accuracy, Recall, and Precision

In the previous lecture note, we learned how to build and interpret a **confusion matrix**.

In this lecture note, we use the confusion matrix to calculate three important **performance metrics**:

- **Accuracy** — overall, how many predictions were correct?
- **Recall** — out of all actual cancer cases, how many did we catch?
- **Precision** — out of all predicted cancer cases, how many were actually cancer?

We continue with the same tumor prediction scenario:

- `0` = not cancer (benign)
- `1` = cancer (malignant)

To make these metrics more meaningful, we use a larger dataset than in the previous lecture note — one that produces a test set with **both false positives and false negatives**.

---
## Review: Confusion Matrix Format

We use the following format throughout this lecture note.
**Rows = actual labels, columns = predicted labels.**

|  | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN | FP |
| **Actual 1** | FN | TP |

So the matrix is:

$$
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
$$

---
## Setup: Load Data and Fit the Model

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

tumor_size = np.array([
    1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8,
    3.0, 3.2, 3.4, 3.6, 3.8, 4.0, 4.2, 4.4, 4.6, 4.8,
    5.0, 5.2, 5.4, 5.6, 5.8, 6.0, 6.2, 6.4, 6.6, 6.8,
    7.0, 7.2, 7.4, 7.6, 7.8, 8.0, 8.2, 8.4, 8.6, 8.8
])

diagnosis = np.array([
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 1, 0, 1, 0, 1,
    0, 1, 1, 0, 1, 1, 1, 0, 1, 1,
    1, 0, 1, 1, 1, 1, 1, 1, 1, 1
])

# 0 = benign, 1 = malignant
X = tumor_size.reshape(-1, 1)
y = diagnosis

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=43
)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Total observations:', len(X))
print('Training samples: ', len(X_train))
print('Test samples:     ', len(X_test))

In [ ]:
print('Tumor sizes in test set:', X_test.ravel())
print('True labels:            ', y_test)
print('Predicted labels:       ', y_pred)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:')
print(cm)

---
## Confusion Matrix for This Example

From the code above, the confusion matrix is:

$$
\begin{bmatrix}
5 & 2 \\
1 & 4
\end{bmatrix}
$$

This means:

- $TN = 5$ — correctly predicted benign
- $FP = 2$ — predicted malignant but actually benign (false alarm)
- $FN = 1$ — predicted benign but actually malignant (missed cancer)
- $TP = 4$ — correctly predicted malignant

This example is more realistic than before: it contains **both** false positives and false negatives, so we can see exactly what each metric is measuring.

---
## Step 1: Accuracy

**Accuracy** measures the proportion of all predictions that were correct.

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Using our example:

$$\text{Accuracy} = \frac{4 + 5}{4 + 5 + 2 + 1} = \frac{9}{12} = 0.75$$

So the model is correct on **75%** of test cases.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

---
## Is Accuracy Alone Enough?

Not always. In this example, accuracy is **75%**, which sounds reasonable.

But the confusion matrix tells us something more important:

- The model made **2 false positives** — 2 healthy patients were flagged as having cancer
- The model made **1 false negative** — 1 real cancer was missed

Accuracy treats these two errors equally — both count as one wrong prediction. But in medicine they have very different consequences.

> **Why accuracy alone can mislead — a simple example:**
>
> Our test set has **5 malignant** and **7 benign** tumors (12 total).
>
> Suppose a model always predicts **benign (0)** for every patient — it does not even look at tumor size.
>
> - For the 7 benign tumors: prediction = 0, true label = 0 → **correct** ($TN = 7$)
> - For the 5 malignant tumors: prediction = 0, true label = 1 → **wrong** ($FN = 5$)
>
> Accuracy = 7 correct out of 12 = **58%**
>
> 58% might seem like a weak model — but accuracy does not tell you **why** it is weak.
> The critical fact — that it missed **every single cancer** — is completely hidden.
>
> This is the core problem: whether the accuracy number is 58% or 75%, accuracy alone cannot distinguish between **missed cancers** and **false alarms**. That is why we need recall and precision.

---
## Step 2: Recall

**Recall** answers the question:

> **Out of all actual cancer cases, how many did the model correctly identify?**

$$\text{Recall} = \frac{TP}{TP + FN}$$

Using our example:

$$\text{Recall} = \frac{4}{4 + 1} = \frac{4}{5} = 0.80$$

So the recall is **0.80** — the model correctly identified **80%** of the actual cancer cases.

In medical settings, **recall is often the most important metric** — because missing a real cancer (a false negative) can be extremely serious.

> **Note:** Recall is also called **sensitivity** in medical contexts.

In [ ]:
from sklearn.metrics import recall_score

recall_score(y_test, y_pred)

---
## Step 3: Precision

**Precision** answers the question:

> **Out of all cases the model predicted as cancer, how many were actually cancer?**

$$\text{Precision} = \frac{TP}{TP + FP}$$

Using our example:

$$\text{Precision} = \frac{4}{4 + 2} = \frac{4}{6} \approx 0.667$$

So the precision is about **0.667** — when the model predicted cancer, it was correct about **two-thirds** of the time.

**High precision does not mean high recall, and vice versa.** These two metrics measure different kinds of success.

In [ ]:
from sklearn.metrics import precision_score

precision_score(y_test, y_pred)

---
## Recall vs. Precision

Recall and precision are not the same — they focus on different things:

| Metric | Focuses on | Question |
|---|---|---|
| **Recall** | Actual cancer cases | Did we catch all the real cancers? |
| **Precision** | Predicted cancer cases | When we said "cancer", were we right? |

In our example:

- **Recall = 0.80** — the model caught 4 out of 5 real cancers
- **Precision ≈ 0.667** — when the model predicted cancer, it was correct 2 out of 3 times

So the model is **better at catching real cancers than it is at avoiding false alarms**.

In cancer screening this is usually the right balance — catching most real cancers matters more than eliminating every false alarm.

---
## All Metrics Together

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('Recall   :', round(recall_score(y_test, y_pred), 3))
print('Precision:', round(precision_score(y_test, y_pred), 3))

---
## Which Metric Matters Most?

There is no single best metric for every situation. It depends on the problem.

| Metric | When it matters most |
|---|---|
| **Accuracy** | When all classes are equally important and the dataset is balanced |
| **Recall** | When missing a positive case is the most dangerous error (e.g. cancer screening) |
| **Precision** | When a false alarm is costly (e.g. fraud detection) |

In our tumor example:

- **Accuracy = 0.750** — looks decent, but hides the type of errors being made
- **Recall = 0.800** — the model catches 80% of real cancers; 1 real cancer was missed
- **Precision = 0.667** — 2 out of 6 cancer predictions were false alarms

For **cancer screening**, recall is usually the most critical metric — a false negative means a real cancer may go untreated.

---
## Practice Problems

Try to answer each question before opening the answer.

---

### Question 1

In this example, recall = 0.80 and precision ≈ 0.667. What does this tell you about how the model behaves?

<details>
<summary>Answer</summary>

The model catches 80% of real cancers — it is fairly good at not missing malignant tumors. However, when it predicts cancer, it is only correct about two-thirds of the time — meaning it raises false alarms on some benign tumors. For cancer screening, this is generally an acceptable tradeoff: catching most real cancers is more important than eliminating every false alarm.

</details>

---

### Question 2

Why is accuracy alone not enough in this tumor example?

<details>
<summary>Answer</summary>

The model achieves 75% accuracy, which sounds reasonable. But accuracy treats all errors equally — a missed cancer and a false alarm both count as one wrong prediction. In reality, a false negative (missed cancer) is far more dangerous than a false positive (false alarm). Accuracy alone would not reveal that the model missed one real cancer and produced two false alarms. Recall and precision expose these weaknesses directly.

</details>

---

### Question 3

Suppose a model always predicts **malignant (1)** for every patient, regardless of tumor size. Using our test set (5 malignant, 7 benign), what would its **recall** and **precision** be? Explain why.

<details>
<summary>Answer</summary>

If the model predicts malignant for all 12 test cases:

- $TP = 5$, $FP = 7$, $FN = 0$
- **Recall** $= 5 / (5 + 0) = 1.0$ — perfect recall, because it never misses a real cancer
- **Precision** $= 5 / (5 + 7) = 5/12 \approx 0.417$ — poor precision, because more than half of its cancer predictions are wrong

This shows that high recall alone does not mean the model is good. A model that cries wolf on every patient catches all cancers, but raises far too many false alarms.

</details>

---

### Question 4

In our example, $FN = 1$ (one missed cancer). If a doctor decides to lower the decision threshold to catch that missed cancer, what is the likely effect on precision?

<details>
<summary>Answer</summary>

Lowering the threshold makes the model more willing to predict malignant — so the boundary shifts and more tumors get flagged as cancer. This usually increases recall (fewer missed cancers) but decreases precision (more false alarms). The model would likely flag some additional benign tumors as malignant, increasing FP and reducing precision.

</details>

---

### Question 5

In this example, which is larger — recall or precision — and why does this make sense for a cancer screening model?

<details>
<summary>Answer</summary>

Recall (0.80) is larger than precision (0.667). This makes sense for cancer screening because the model is designed to be cautious — it would rather raise a false alarm (which can be resolved with further testing) than miss a real cancer (which could be life-threatening if untreated). A recall higher than precision reflects this cautious behaviour.

</details>

---

### Question 6

Why can a high accuracy score be misleading in a cancer screening setting?

<details>
<summary>Answer</summary>

Because accuracy does not distinguish between types of errors. If a dataset has many more benign cases than malignant cases, a model that rarely predicts malignant could achieve high accuracy simply by always predicting benign — while missing most real cancers. In cancer screening, what matters most is whether real cancers are caught (recall) and how reliable the cancer predictions are (precision), not just the overall proportion of correct predictions.

</details>

---
## Summary

| Metric | Formula | Question it answers |
|---|---|---|
| **Accuracy** | $\frac{TP + TN}{TP + TN + FP + FN}$ | Overall, how many predictions were correct? |
| **Recall** | $\frac{TP}{TP + FN}$ | Out of all actual cancer cases, how many did we catch? |
| **Precision** | $\frac{TP}{TP + FP}$ | Out of all predicted cancer cases, how many were actually cancer? |

**Key results for our model** (40 observations, 12 in test set):

| | Value | Notes |
|---|---|---|
| TP | 4 | Correctly predicted malignant |
| TN | 5 | Correctly predicted benign |
| FP | 2 | False alarms |
| FN | 1 | Missed cancer |
| **Accuracy** | **0.750** | Looks decent — but hides the types of errors |
| **Recall** | **0.800** | 4 out of 5 real cancers were caught |
| **Precision** | **0.667** | 4 out of 6 cancer predictions were correct |

> **Key lesson:** Accuracy alone is not enough. Recall and precision reveal **which kind of errors** the model makes — and in cancer diagnosis, not all errors are equal.

In a later lecture note, we will study how changing the **decision threshold** shifts the balance between recall and precision.